# LED OCR ablation, phase 2: extract pipeline inputs (Colab T4, resumable)
Protocol `led-ocr-ablation-261110-v1`. Smoke set first (`SET = 'smoke'`), read the timings, then `SET = 'cohort'` (40 videos, about 10 to 11 GPU hours, usually several sessions).

**How to resume after a session ended or disconnected**
1. Open this notebook again and run the setup, install and helper cells (the first three code cells). `/content` is wiped between sessions, so the install repeats; your progress is not lost.
2. Run the **status** cell: it shows done / error / left and an ETA, read from the ledger on Drive.
3. Run the **extract** cell. It skips every video already finished (ledger row plus its input file on Drive) and continues with the first unfinished one.

Everything that matters lives on Drive under `WORK`: `videos/` (downloaded videos), `run/inputs/` (one input file per finished video), `run/extract_ledger_<set>.jsonl` (the checkpoint), `run/progress_<set>.json`, `run/logs/`. A video interrupted mid-run is processed again from its start (about 20 minutes lost at most).

Set `MAX_MINUTES` in the extract cell (for example 600) to stop cleanly before a session limit. A pipeline error is final unless you list its id in `RETRY_IDS` after an infrastructure failure. A failed download or a Qwen3-ASR fallback writes no row, so the next run tries that video again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, subprocess
from pathlib import Path

APP = '/content/multimodal-lecture-summarizer'   # app repo (separate repo)
BENCH = '/content/benchmark'                     # benchmark repo (separate repo, private)
WORK = '/content/drive/MyDrive/led-ocr-ablation-261110'   # private: videos/, run/
SET = 'smoke'                                    # 'smoke' first, then 'cohort'
APP_COMMIT = '5e26ea799de9ac7e5a934d684ed4a031ff1202e4'  # frozen in protocol.json
PY = '/content/venv/bin/python'                  # same layout as the app's Colab worker notebook
PROTOCOL = f'{BENCH}/experiments/registry/protocols/led-ocr-ablation-261110-v1'

# benchmark repo is private: add a Colab secret GITHUB_TOKEN (read access) and enable notebook access.
try:
    from google.colab import userdata
    _tok = userdata.get('GITHUB_TOKEN')
except Exception:
    _tok = ''

def _clone(repo, dest):
    if os.path.isdir(dest): return
    base = f'x-access-token:{_tok}@' if _tok else ''
    r = subprocess.run(['git', 'clone', '-q', f'https://{base}github.com/multimodal-lecture-summarizer/{repo}.git', dest],
                       env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    assert r.returncode == 0, f'clone of {repo} failed (private repo? set the GITHUB_TOKEN Colab secret)'

_clone('benchmark', BENCH); _clone('multimodal-lecture-summarizer', APP)
subprocess.run(['git', '-C', APP, 'checkout', '-q', APP_COMMIT], check=True)  # the scripts refuse any other app commit
os.makedirs(f'{WORK}/videos', exist_ok=True); os.makedirs(f'{WORK}/run', exist_ok=True)

hashes = json.load(open(f'{PROTOCOL}/hashes.json'))['sha256']
for name, expected in hashes.items():
    assert hashlib.sha256(open(f'{PROTOCOL}/{name}', 'rb').read()).hexdigest() == expected, name
print('frozen files verified')

In [ ]:
# Python 3.11 venv as in the app's Colab worker; Colab's own Python has no torch 2.5.1 wheels.
if not Path(PY).is_file():
    subprocess.run(['uv', 'venv', '--python', '3.11', '/content/venv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '-r', f'{APP}/ai_workers/requirements.txt',
                '-r', f'{APP}/backend/requirements.txt', 'setuptools'], check=True, cwd=APP)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--no-deps', '-r', f'{APP}/ai_workers/requirements-tracking.txt'], check=True, cwd=APP)
subprocess.run(['nvidia-smi', '-L'])

In [ ]:
site = '/content/venv/lib/python3.11/site-packages'
ENV = {**os.environ,
       'LD_LIBRARY_PATH': f'{site}/nvidia/cudnn/lib:{site}/nvidia/cublas/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
       'PATH': '/content/venv/bin:' + os.environ['PATH'],
       'PADDLEOCR_USE_GPU': '0', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8',
       'PROCESS_MIN_AVAILABLE_MEMORY_MB': '4096', 'PROCESS_SOFT_MIN_AVAILABLE_MEMORY_MB': '3072',
       'HF_HOME': '/content/hf'}

import time, datetime
LOG_DIR = Path(f'{WORK}/run/logs'); LOG_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None, tag='step'):
    """Stream output to the cell, tee it to a Drive log file, print timing, stop on a non-zero exit."""
    logfile = LOG_DIR / f'{SET}-{tag}-{datetime.datetime.now():%Y%m%d-%H%M%S}.log'
    print(f'$ {" ".join(map(str, cmd))}\n  cwd={cwd}  log={logfile}', flush=True)
    started = time.time()
    p = subprocess.Popen(cmd, cwd=cwd, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(logfile, 'w', encoding='utf-8') as out:
        for line in p.stdout:
            print(line, end=''); out.write(line); out.flush()
    code = p.wait()
    print(f'\n[{tag}] exit code {code} after {time.time() - started:.0f}s; full log: {logfile}')
    assert code == 0, f'{tag} failed with exit code {code}; read the output above (tracebacks also in {LOG_DIR}/errors.log)'

def show_errors(n=60):
    f = LOG_DIR / 'errors.log'
    print(''.join(f.read_text(encoding='utf-8').splitlines(True)[-n:]) if f.exists() else 'no errors.log (no failed video/cell)')

In [ ]:
# STATUS: no GPU or app venv needed. Run it before and after the extract cell.
run(['python', '-m', 'experiments.scripts.led_ocr_ablation_extract', '--status', '--app-root', APP, '--videos-dir', f'{WORK}/videos',
     '--out-dir', f'{WORK}/run', '--set', SET, '--protocol-dir', PROTOCOL], cwd=BENCH, env=dict(os.environ), tag='status')

In [ ]:
# EXTRACT: resumable; rerun it in every session until the status shows 0 left.
MAX_MINUTES = 0     # minutes this cell may run (counted from its start, not from the session start: subtract the ~15 min of setup). 0 = no budget
RETRY_IDS = []      # only for videos whose ledger row is an error caused by infrastructure (OOM, killed session). Each id is retried once; clear the list afterwards.
cmd = [PY, '-m', 'experiments.scripts.led_ocr_ablation_extract', '--app-root', APP, '--videos-dir', f'{WORK}/videos',
       '--out-dir', f'{WORK}/run', '--set', SET, '--protocol-dir', PROTOCOL, '--max-minutes', str(MAX_MINUTES)]
if RETRY_IDS:
    cmd += ['--retry-ids', *RETRY_IDS]
run(cmd, cwd=BENCH, tag='extract')

In [ ]:
show_errors()  # tracebacks of failed videos or downloads, if any

In [ ]:
# Descriptive tables only: by stratum, OCR volume per video, segmenter fallbacks. Not used to drop videos.
import sys, pandas as pd
sys.path.insert(0, BENCH)
from experiments.scripts.led_ocr_ablation_extract import read_rows
rows = read_rows(Path(f'{WORK}/run/extract_ledger_{SET}.jsonl'))
df = pd.DataFrame(rows).drop_duplicates('id', keep='last') if rows else pd.DataFrame()
if len(df):
    cols = [c for c in ['id','stratum','status','segmenter','scenes','keyframes_with_ocr','raw_ocr_words','merge_ocr_words','merge_slides_used','merge_transcript_words','asr_model','wall_seconds','error','detail','retry_of'] if c in df]
    display(df[cols])
    if 'segmenter' in df: print('segmenter counts:', df['segmenter'].value_counts(dropna=False).to_dict())
    if 'wall_seconds' in df: print('total wall hours so far:', round(df['wall_seconds'].fillna(0).sum() / 3600, 2))
else:
    print('no ledger rows yet')